# Postprocess SWOT CalVal Data

This notebook continues the CalVal workflow after the download step in `ex_aviso_download_swot_Svenja.ipynb`.

## What this notebook does

- Loads CalVal passes `9` and `22` for the Mid-Atlantic Bight (MAB) calibration/validation phase
- Processes cycles in the range **478–579**
- Builds a consistent dataset ready for plotting and analysis

## Processing steps

1. Locate files for the selected passes in the data directory
2. Load each file and concatenate them along a new `cycle` dimension
3. Derive absolute dynamic topography (`ADT`) from `MDT + SSHA`
4. Compute filtered speed from rotated SWOT velocity components
5. Extract and normalize cycle metadata for consistent time indexing


In [ ]:
import sys
from pathlib import Path
sys.path.append("/home/sryan/python/") # go to parent dir
from utils.plot_utils import finished_plot,latlon_label
from utils.datafun import datestr2doy,datetime2numeric,numeric2datetime
from utils.vector_fun import rotate,compute_angle
from utils.general import pickle_save,pickle_load

# load SWOT diagnoctis 
# add the directory *above* the SwotDiag/ folder
sys.path.append("/home/sryan/python/projects/NASA_SWOT_NWA_shelf_2024/SwotDiag")
from SwotDiag.diagnosis import *

# load project startup file with all relevant functions
exec(open('/home/sryan/python/projects/NASA_SWOT_NWA_shelf_2024/startup_swot.py').read())

# path for saving calval plots
plotpath = './projects/NASA_SWOT_NWA_shelf_2024/analysis/plots/CalVal/Antarctica/'

In [ ]:
# ----------------------------------------
# open dataset
# ----------------------------------------
filepath = '/srv/data/SWOT/L3/CalVal/v3_0/'

# 1) find files of specific passes (9, 22) in directory
directory = Path(filepath)
# matches = list(directory.glob(f"*Expert_{cycle:003d}_*.nc"))
matches9 = sorted(directory.glob(f"*Expert_???_{9:003d}_*.nc"))#,concat_dim="time")
matches22 = sorted(directory.glob(f"*Expert_???_{22:003d}_*.nc"))#,concat_dim="time")

# 2) load and concatenate along new dimension 'cycle'
# PassID9
datasets9 = [] 
for f in matches9:
    dummy = xr.open_dataset(f)
    if 'i_num_pixel' in dummy:
        dummy = dummy.drop_vars(['i_num_pixel','i_num_line'])
    datasets9.append(dummy)

# PassID 22
datasets22 = [] 
for f in matches22:
    dummy = xr.open_dataset(f)
    if 'i_num_pixel' in dummy:
        dummy = dummy.drop_vars(['i_num_pixel','i_num_line'])
    datasets22.append(dummy)
    
# concatenate
pass9 = xr.concat(datasets9,dim="cycle")
pass22 = xr.concat(datasets22,dim="cycle")

# 3) derive ADT (MDT+SSHA) and speed and add to dataset
# add absolute dynamic topography
pass9['adt_filtered'] = pass9['mdt'] + pass9['ssha_filtered']
pass22['adt_filtered'] = pass22['mdt'] + pass22['ssha_filtered']
pass9['adt_unfiltered'] = pass9['mdt'] + pass9['ssha_unfiltered']
pass22['adt_unfiltered'] = pass22['mdt'] + pass22['ssha_unfiltered']

#--------------------------------------------
# add curred speed
pass9['speed_filtered'] = np.sqrt(pass9['ugos_filtered']**2 +  pass9['vgos_filtered']**2)
pass22['speed_filtered'] = np.sqrt(pass22['ugos_filtered']**2 +  pass22['vgos_filtered']**2)

# 4) extract cycle numbers from filenames and add as variable
def extract_cycle(files):
    import re
    cycle = []
    for file in files:
        m = re.search(r"Expert_(\d{3})", str(file))
        if not m:
            raise ValueError("Filename does not match expected pattern")
        # Extract as strings or ints
        cycle.append(int(m.groups()[0]))  
    return cycle

cycles9 = extract_cycle(matches9)
cycles22 = extract_cycle(matches22)
pass9["cycle"] = (('cycle'),cycles9)
pass22["cycle"] = (('cycle'),cycles22)



In [ ]:


# 5) adjust for missing cycles and bring to same format
#**Problem**
#I want to bring both passes ontp same time grid. Ideally I am interpolated using the cycles, however normal interp will not interpolate a datetime object. Hence I am adding an additional numeric time variable which can be inteprolated and convert back to datetime
pass22['timenum'] = (('cycle','num_lines'), datetime2numeric(pass22.time,unit='s'))
# pass22 = pass22.set_coords("cycle")
pass22 = pass22.reindex(cycle=np.arange(474,578))#swap_dims({'time_cycle':'cycle'})
# now add time variable back
pass22['time'] = (('cycle','num_lines'),numeric2datetime(pass22.timenum,unit='s'))
# set cycle time as coordinate
pass22 = pass22.assign_coords(time_cycle=("cycle",pass22.time.mean(dim='num_lines').values))
pass22 = pass22.swap_dims({'cycle':'time_cycle'})


pass9['timenum'] = (('cycle','num_lines'), datetime2numeric(pass9.time,unit='s'))
pass9 = pass9.reindex(cycle=np.arange(474,578))
# now add time variable back
pass9['time'] = (('cycle','num_lines'),numeric2datetime(pass9.timenum,unit='s'))
# set cycle time as coordinate
pass9 = pass9.assign_coords(time_cycle=("cycle",pass9.time.mean(dim='num_lines').values))
pass9 = pass9.swap_dims({'cycle':'time_cycle'})

# limit latitude bounds to reduce data size --> not necessary to do processing on all points along the swath
pass9 = pass9.where((pass9.latitude>30) & (pass9.latitude<55),drop=True)
pass22 = pass22.where((pass22.latitude>30) & (pass22.latitude<55),drop=True)

Let's look at the dataset created:

In [ ]:
pass9

## Compute diagnostics

Computing diagnostics with SWOT_diag package from Yann-Treden Trachant https://github.com/treden/SwotDiag/tree/main

Codes is using 2D Kernel to smooth data before applying derivatives. Quantities derived are:
- geostrophic velocities
- cyclogeostrophic velociies
- relative vorticity ($\zeta/f$)
- strain rate (Normal: $s_n=\partial u/\partial x - \partial v/\partial y$
  Shear: $s_s=\partial v/\partial x - \partial u/\partial y$)     - stretching and shearing of water without rotation
- Okubo-Weiss Parameter (W=S$^2$-$\zeta^2$)  - determines whether rotation or deformation dominates in given region

### Derive diagnostics - kernel

In [ ]:
###### load saved data (from cell below)
# save data
pass9_diag = np.load('/home/sryan/python/projects/NASA_SWOT_NWA_shelf_2024/derived_data/pass9_diag_calval.nc.npy',allow_pickle=True)
pass22_diag = np.load('/home/sryan/python/projects/NASA_SWOT_NWA_shelf_2024/derived_data/pass22_diag_calval.nc.npy',allow_pickle=True)

In [7]:
%%time
def swot_add_diagnostics(ds,n=5):

    # 1) Define parametes
    params = dict(derivative = 'fit', # Derivative obtained by the fitting method instead of point difference, point 'dxdy'
    n = n, # 13*13 point kernels
    min_valid_points = 0.75, # Ratio of valid points to compute the derivative (e.g. 75% valid points are necessary to compute the derivatives, useful to avoid boundary effects)
    cyclostrophy = 'GW', # Cyclogeostrophic currents are computed using the wind-gradient balance
    avoid_negative = False, # Parameter that avoid negative values in the SQRT using the GW formulation (leading to invalid values in the solutions)
    second_derivative = 'dxdy', # The second derivative is obtained by point-difference the first derivative (obtained by surface fitting method), rather than from the surface curvature. I noticed that it reduces the noise. 
    kernel = 'circular', # can be circular or a square, the shape of the kernel
                 )

    # 2) Diagnistics only run for one timestep
    # --> loop through cycles and then concatenate
    diag = []
    for i in range(len(ds.time_cycle)):
        diag.append(compute_ocean_diagnostics_from_eta(ds.adt_unfiltered.isel(time_cycle=i),
                                                       ds.longitude, ds.latitude, **params))


    # 3) create dictionary with merged variables to then add to xarray
    diag_dic = {}
    for var in diag[0].keys(): # loop over variables in diag
        dummy = []
        for i in range(len(ds.time_cycle)): # loop over cycles
            dummy.append(diag[i][var])
        diag_dic[var] = dummy


    # 4) Adding the computed diagnostics to the original xarray dataset
    dims = ds.ugos_filtered.dims
    ds = ds.assign(ugos = (dims, diag_dic['ug']), vgos = (dims, diag_dic['vg']), # geostrophic currents (eastwar/northward)
                         ucgos = (dims, diag_dic['ucg']), vcgos = (dims, diag_dic['vcg']), # cyclo-geostrophic currents (eastward/northwar)
                         zeta = (dims, diag_dic['zeta']), # relative vorticity
                         sr = (dims, diag_dic['S']), # strain rate
                         OW = (dims, diag_dic['OW'])) # Okubo_Weiss parameter

    # 5) return xarray with new variables
    return ds[['ugos','vgos','ucgos','vcgos','zeta','sr','OW']]

# ----------------------------------------------------------------------------------------
# run for different filter length
pass9_diag ={}
pass22_diag ={}
for nfilt in [5,9,13]:
    pass9_diag[f'{nfilt}p'] = swot_add_diagnostics(pass9,n=nfilt)
    pass22_diag[f'{nfilt}p'] = swot_add_diagnostics(pass22,n=nfilt)
    # add speed
    pass9_diag[f'{nfilt}p']['Ug'] = np.sqrt(pass9_diag[f'{nfilt}p']['ugos']**2 + pass9_diag[f'{nfilt}p']['vgos']**2)
    pass9_diag[f'{nfilt}p']['Ucg'] = np.sqrt(pass9_diag[f'{nfilt}p']['ucgos']**2 + pass9_diag[f'{nfilt}p']['vcgos']**2)
    pass22_diag[f'{nfilt}p']['Ug'] = np.sqrt(pass22_diag[f'{nfilt}p']['ugos']**2 + pass22_diag[f'{nfilt}p']['vgos']**2)
    pass22_diag[f'{nfilt}p']['Ucg'] = np.sqrt(pass22_diag[f'{nfilt}p']['ucgos']**2 + pass22_diag[f'{nfilt}p']['vcgos']**2)

# save data
pickle_save('pass9_diag_calval.nc','/home/sryan/python/projects/NASA_SWOT_NWA_shelf_2024/derived_data/',pass9_diag)
pickle_save('pass22_diag_calval.nc','/home/sryan/python/projects/NASA_SWOT_NWA_shelf_2024/derived_data/',pass22_diag)

KeyboardInterrupt: 

### Add dxdy diagnostics to merged data

In [ ]:
%%time
# derive vorticity, strain and OW from product filtered velocity

def swot_add_diagnostics_filtered(ds):
    # 1) Define parametes
    params = dict(derivative = 'dxdy', # Derivative obtained by the fitting method instead of point difference, point 'dxdy'
    n = 3, # 13*13 point kernels
    min_valid_points = 0.75, # Ratio of valid points to compute the derivative (e.g. 75% valid points are necessary to compute the derivatives, useful to avoid boundary effects)
    cyclostrophy = 'GW', # Cyclogeostrophic currents are computed using the wind-gradient balance
    avoid_negative = False, # Parameter that avoid negative values in the SQRT using the GW formulation (leading to invalid values in the solutions)
    second_derivative = 'dxdy', # The second derivative is obtained by point-difference the first derivative (obtained by surface fitting method), rather than from the surface curvature. I noticed that it reduces the noise. 
    kernel = 'circular', # can be circular or a square, the shape of the kernel
                 )
    
    # 2) Diagnistics only run for one timestep
    # --> loop through cycles and then concatenate
    diag = []
    for i in range(len(ds.time_cycle)):
        diag.append(compute_ocean_diagnostics_from_eta(ds.adt_filtered.isel(time_cycle=i),
                                                       ds.longitude, ds.latitude, **params))
    
    
    # 3) create dictionary with merged variables to then add to xarray
    diag_dic = {}
    for var in diag[0].keys(): # loop over variables in diag
        dummy = []
        for i in range(len(ds.time_cycle)): # loop over cycles
            dummy.append(diag[i][var])
        diag_dic[var] = dummy
    
    # 4) Adding the computed diagnostics to the original xarray dataset
    dims = ds.ugos_filtered.dims
    ds = ds.assign(ugos_derived = (dims, diag_dic['ug']), vgos_derived = (dims, diag_dic['vg']), # geostrophic currents (eastwar/northward)
                         ucgos_filtered = (dims, diag_dic['ucg']), vcgos_filtered = (dims, diag_dic['vcg']), # cyclo-geostrophic currents (eastward/northwar)
                         #Ug = (dims, np.sqrt(diag_dic['ug']**2+diag_dic['vg']**2), # geostrophic speed
                         #Ucg_filtered = (dims, np.sqrt(diag_dic['ucg']**2+diag_dic['vcg']**2)), # cycle-geostrophic speed
                         zeta_filtered = (dims, diag_dic['zeta']), # relative vorticity
                         sr_filtered = (dims, diag_dic['S']), # strain rate
                         OW_filtered = (dims, diag_dic['OW'])) # Okubo_Weiss parameter

    # 5) add speed
    ds['Ug_filtered'] = np.sqrt(ds['ugos_derived']**2 + ds['vgos_derived']**2)
    ds['Ucg_filtered'] = np.sqrt(ds['ucgos_filtered']**2 + ds['vcgos_filtered']**2)
    
    # 5) return xarray with new variables
    return ds

# add diganostics to original dataset
pass9 = swot_add_diagnostics_filtered(pass9)
pass22 = swot_add_diagnostics_filtered(pass22)

In [ ]:
# --------------------------------------------------
# save data for easier future use
pass9.to_netcdf('/srv/data/SWOT/L3/derived_data/pass9_CalVal_v2.nc')
pass22.to_netcdf('/srv/data/SWOT/L3/derived_data/pass22_CalVal_v2.nc')